# 02 · Buildings, 3D massing and population

**Question:** How do building assumptions change a synthetic population allocation?

**Runtime:** Python / Pyodide in JupyterLite. Choose **Run → Run All Cells**.
Each notebook is independent; start with a fresh kernel. Initial runtime/package downloads need internet.
The initial baseline uses **synthetic educational fixtures** near a Gaborone-area anchor.
Extended investigations explicitly identify bundled real OSM, Sentinel or terrain extracts where used;
these do not validate the synthetic population, mobility, exposure or scenario assumptions.

**Astra experiment:** Have Astra critique floor-area proxies and design a conservation test; compare the suggested allocation to the exact baseline.
Run the numerical baseline first, then give Astra the exported evidence and the prompt below.
The baseline does not call or run GPT-6 Astra. No API key is needed.


In [ ]:
import sys, json, math, time
from pathlib import Path
if sys.platform == "emscripten":
    import piplite
    await piplite.install(["numpy", "matplotlib"])
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from twin import *
style()
print("Runtime:", sys.platform, "| data: synthetic | seed: 42")


## Allocate people using residential floor area
Change `population_total` or `floor_height_m` and rerun. Integer largest-remainder allocation conserves population.
Floor counts and occupancy are invented; height is a visualization assumption.


In [ ]:
c=city(); population_total=1200; floor_height_m=3.
pop=allocate(c['capacity'],population_total)
flat=allocate(c['width']*c['depth']*c['residential'],population_total)
assert pop.sum()==population_total and (pop[~c['residential']]==0).all()
assert (pop>=0).all()
print('Population conserved:',pop.sum(),'| people reassigned when ignoring floors:',int(abs(pop-flat).sum()/2))


In [ ]:
fig=plt.figure(figsize=(12,5)); ax=fig.add_subplot(121,projection='3d')
ax.bar3d(c['xy'][:,0],c['xy'][:,1],np.zeros(len(pop)),c['width'],c['depth'],c['floors']*floor_height_m,
         color=np.where(c['residential'],'#087f8c','#f2a541'),shade=True)
ax.set(xlabel='East (m)',ylabel='North (m)',zlabel='Height (m)',title='Synthetic 3D massing')
ax=fig.add_subplot(122); dots=ax.scatter(*c['xy'].T,c=pop,s=35,cmap='viridis')
map_axes(ax,'Allocated residents'); fig.colorbar(dots,ax=ax,label='People'); plt.tight_layout(); plt.show()


In [ ]:
rows=[dict(building=f'b{i:03d}',population=int(v),floor_area_m2=float(c['capacity'][i])) for i,v in enumerate(pop)]
export_json('02_population.json',dict(synthetic=True,total=population_total,buildings=rows))
print('Astra prompt: Explain why a floor-area-weighted population surface does not establish household size, poverty, or disease risk. Propose three external validation datasets.')


## Teaching lab: Challenge an allocation
**Before using AI:** write a prediction, run the numerical baseline, and explain one surprising result.

**Astra evidence:** send only 02_population.json and the stated allocation assumptions. Use a fresh conversation and record the exact model,
date, interface, tools and prompt. A task with a supplied result tests interpretation, not independent calculation.

**Copyable Astra prompt:**
> Distinguish known totals from invented allocation weights. Propose a sensitivity experiment and state which household-level claims cannot be inferred.

**Implementation brief — review the runnable extension below:**
> Compare population allocation under floor counts of one versus the generated floors. Report the number of people reassigned. Edit the authoring source in `scripts/make_notebooks.py` (or
> `scripts/teaching_labs.py` for labs 11–13), regenerate the notebooks and preserve browser compatibility.
> Use NumPy, Matplotlib and the standard library. Include a small reference case that can be checked by hand.

**Acceptance evidence:** Both allocations sum to 1200; nonresidential buildings get zero; a change in assumptions is not new census evidence.

The extended investigation later in this notebook implements this line of work with maps and checks.
Use the brief to review the implementation or ask Codex for a further controlled variation.

**Share back:** save your original prediction, changed parameter, result, model response and one limitation.
Download the edited notebook and exports. If you have no model access, exchange explanations with a partner
and evaluate them with the same criteria; do not label that activity as an Astra trial.


## Extended investigation: Constrain neighborhoods and map allocation uncertainty

Allocate independently within four geographic quadrants, then compare single-floor, generated-floor and taller-building assumptions. Every scenario conserves each neighborhood total. The spread describes sensitivity to assumptions, not a confidence interval or observed household occupancy. Real OSM height fields are inspected separately: missing height remains unknown.

**Read the concept map first:** identify the input, method, evidence and limit. Predict the spatial pattern before running the comparison.


In [ ]:
from extension_tools import concept_map, export_investigation
concept_map('Constrain neighborhoods and map allocation uncertainty', ['Four invented area totals', 'Allocate within each area', 'Conservation + range maps', 'Floors are not occupants'])
plt.show()


In [ ]:
from extension_tools import load_data
ex_c=city(); ex_groups=(ex_c['xy'][:,0]>=1000).astype(int)+2*(ex_c['xy'][:,1]>=1000)
ex_totals=np.array([300,250,400,250]); ex_floor_options=[np.ones(144),ex_c['floors'],ex_c['floors']+2]
ex_allocations=[]
for floors in ex_floor_options:
    allocation=np.zeros(144,dtype=int)
    for group,total in enumerate(ex_totals):
        mask=ex_groups==group
        allocation[mask]=allocate(ex_c['width'][mask]*ex_c['depth'][mask]*floors[mask]*ex_c['residential'][mask],int(total))
        assert allocation[mask].sum()==total
    assert allocation.sum()==1200 and not allocation[~ex_c['residential']].any()
    ex_allocations.append(allocation)
ex_allocations=np.array(ex_allocations); ex_spread=np.ptp(ex_allocations,axis=0)
ex_reassigned=int(abs(ex_allocations[1]-ex_allocations[0]).sum()//2)
print('Minimum residents reassigned between flat and generated-floor scenarios:',ex_reassigned)
fig,axes=plt.subplots(1,3,figsize=(14,4))
for ax,values,title,cmap in zip(axes,[ex_groups,ex_allocations[1]-ex_allocations[0],ex_spread],['Neighborhood constraints','People reassigned: floors minus flat','Range across floor assumptions'],['tab10','coolwarm','magma']):
    options={}
    if title.startswith('People'):
        limit=max(1,int(abs(values).max())); options=dict(vmin=-limit,vmax=limit)
    dots=ax.scatter(*ex_c['xy'].T,c=values,cmap=cmap,s=30,**options); map_axes(ax,title)
    colorbar=fig.colorbar(dots,ax=ax,label='Area ID' if title.startswith('Neighborhood') else 'People')
    if title.startswith('Neighborhood'): colorbar.set_ticks([0,1,2,3])
    ax.axvline(1000,color='gray',lw=.6); ax.axhline(1000,color='gray',lw=.6)
fig.tight_layout(); extension_figure=fig
ex_osm=load_data('gaborone_buildings.geojson'); ex_missing=sum(f['properties']['height_m'] is None for f in ex_osm['features'])
print('Scenario neighborhood totals:',[[int(a[ex_groups==g].sum()) for g in range(4)] for a in ex_allocations])
print('Real OSM features without reported height:',ex_missing,'— do not invent measured heights.')
extension_evidence=dict(neighborhood_totals=ex_totals.tolist(),population_by_scenario=ex_allocations.tolist(),group=ex_groups.tolist(),units='people',minimum_reassigned=ex_reassigned,floor_scenarios=['one','generated','generated plus two'])
extension_expected=dict(total_population=1200,max_building_range=int(ex_spread.max()))
extension_task='Verify population conservation and calculate the largest per-building population range across scenarios. Return total_population and max_building_range. The range is assumption sensitivity, not a statistical confidence interval.'


In [ ]:
extension_prompt,extension_reference=export_investigation('02',extension_evidence,extension_expected,extension_task,extension_figure)
plt.show()
print('Send only 02_extension_prompt.json and optionally 02_extension_map.png to Astra. Keep the reference in this kernel.')


### Ask, check and explain the spatial result
Download `02_extension_prompt.json` and `02_extension_map.png`. Send the evidence in a fresh Astra
session with the exact task and output fields. For a separate image-only experiment, supply only the image
and task wording, record that condition, and allow clarification if the image does not contain enough information.
Never send the reference or this complete notebook for a blind trial.

Paste the unedited response below or upload a UTF-8 response file. The default is deliberately empty:
**no model trial has run**. After recording a real answer, set `extension_source='MODEL'` and complete the metadata.
The numeric tolerance is 0.01 in each requested field's declared units; labels and booleans require exact agreement.
Keep that rule fixed before examining a response. Inspect the explanation separately for unsupported claims.

**Visual reflection:** point to one map pattern, cite the numerical check that supports it, and name one thing
the visual cannot establish. Re-run one parameter change and preserve both maps. The implementation above
is available for inspection and modification; it is not a measured claim about model accuracy.


In [ ]:
from experiments import grade_answer
extension_source='NOT RUN'
extension_model=''; extension_date=''; extension_interface=''; extension_condition='text-and-map'
extension_response_file=''; extension_response_text=''
if extension_response_file:
    extension_response_text=Path(extension_response_file).read_text(encoding='utf-8')
if extension_source=='MODEL':
    assert extension_model and extension_date and extension_interface and extension_response_text, 'Record the actual model, date, interface and response'
extension_tolerances={key:.01 for key,value in extension_reference.items() if type(value) in (int,float)}
extension_grade=grade_answer(extension_response_text,extension_reference,extension_tolerances) if extension_response_text else dict(status='NOT RUN')
display(extension_grade)
export_json('02_extension_trial.json',dict(source=extension_source,model=extension_model,date=extension_date,
    interface=extension_interface,condition=extension_condition,prompt=extension_prompt,response_text=extension_response_text,
    tolerances=extension_tolerances,reference=extension_reference,grade=extension_grade))


## Save the investigation
Download the edited notebook, evidence, map and trial record from `exports/`. Browser storage does not
commit changes to GitHub. Keep data attribution and the distinction between observed data, synthetic
assumptions and actual model responses when sharing. The extended code and its checks above are part of
this notebook; use them as the starting point for your next controlled comparison.
